# PACT-18 · 5 万条 × 8 小时（Kaggle T4×2）

生产用 notebook：一次跑完 **数据混合 → 双卡 QLoRA（4 折 + 重训）→ 经典成员 → 后处理 → submission.parquet**。

- Accelerator：**GPU T4 × 2**（只选单卡会直接退出并打印可见 GPU 数）
- Internet：**Off**
- 实测吞吐：**0.216 s/样本（双卡合计，1.7B / batch 8 / 384 tok）**
- 预计墙钟：**7.8–8.4 h**（3 折 × 2.0 h + 重训 2.7 h + CPU 阶段 0.3 h），`--deadline-hours 8.8` 兜底

需要挂载：

| 类型 | 引用 |
|---|---|
| Competition | `name-that-llm` |
| Dataset | `jokercarter/ntl-pact18-verify`（源码 + wheels + 外部语料） |
| Dataset | `vodung020905/qwen3-1-7b`（Apache-2.0 权重） |


In [ ]:
import glob, os, shutil, subprocess, sys, time
import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, '| cuda', torch.version.cuda)
print('devices :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | cc {p.major}.{p.minor}')
print('input   :', sorted(os.listdir('/kaggle/input')))


In [ ]:
def pick_dir(patterns, marker):
    for pat in patterns:
        for cand in sorted(glob.glob(pat)):
            if os.path.isdir(cand) and os.path.exists(os.path.join(cand, marker)):
                return cand
    return None

DS = pick_dir(['/kaggle/input/datasets/*/ntl-pact18-verify',
               '/kaggle/input/ntl-pact18-verify'], 'train_verify.csv')
MODEL = pick_dir(['/kaggle/input/datasets/*/qwen3-1-7b',
                  '/kaggle/input/qwen3-1-7b'], 'config.json')
COMP = pick_dir(['/kaggle/input/competitions/name-that-llm',
                 '/kaggle/input/name-that-llm'], 'test.csv')
print('DS   :', DS)
print('MODEL:', MODEL)
print('COMP :', COMP)
assert DS and MODEL and COMP, 'Add Input: competition + 2 datasets'

!pip install -q --no-index --find-links {DS} bitsandbytes 2>&1 | tail -2
import bitsandbytes as bnb
print('bitsandbytes', bnb.__version__)


In [ ]:
WORK = '/kaggle/working/ntl-pact18-code'
if not os.path.isdir(WORK):
    zip_path = f'{DS}/ntl-pact18-code.zip'
    if os.path.exists(zip_path):
        subprocess.call(f'cd /kaggle/working && unzip -oq {zip_path}', shell=True)
    else:
        hits = glob.glob(f'{DS}/**/src/ntl_pact18/pipeline.py', recursive=True)
        assert hits, f'no package under {DS}'
        shutil.copytree(hits[0].split('/src/ntl_pact18/')[0], WORK)
%cd {WORK}
!python -m pytest -q tests --ignore=tests/test_pipeline_smoke.py 2>&1 | tail -3


In [ ]:
# 2,699 竞赛行 + 47,304 外部行（其中 3,430 条为尾部类重复），带 origin 列
rc = subprocess.call(f'''python scripts/build_external_mix.py \
  --base {COMP}/train.csv \
  --external {DS}/ntl_merged_external_large.parquet \
  --target-total 50000 --oversample \
  --out /kaggle/working/trainval_plus_ext_50k.csv''', shell=True)
print('build exit', rc)


In [ ]:
t0 = time.time()
rc = subprocess.call(f'''python scripts/train_parallel.py --stage all \
  --config configs/kaggle_8h_50k_3fold.yaml \
  --deadline-hours 8.8 \
  --reserve-hours 0.55 \
  --train-file /kaggle/working/trainval_plus_ext_50k.csv \
  --model-path {MODEL} \
  --test-file {COMP}/test.csv \
  --submission /kaggle/working/submission.parquet''', shell=True)
print('\n[exit]', rc, 'in', round((time.time() - t0) / 3600, 2), 'h')


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from ntl_pact18.labels import LABELS

sub = pd.read_parquet('/kaggle/working/submission.parquet')
test = pd.read_csv(f'{COMP}/test.csv')
print('shape     :', sub.shape)
print('columns   :', list(sub.columns))
print('nulls     :', int(sub.isna().sum().sum()), '| dup ids:', int(sub['id'].duplicated().sum()))
print('labels ok :', set(sub['model']).issubset(set(LABELS)))
print('rows match:', len(sub) == len(test))
print(sub['model'].value_counts().to_string())


## 备注

- 隐藏测试集约 4,000 行，这里交互态只有 3 行，**行数不同是正常的**。
- 真正提交要在 Code 页点 **Submit to Competition**，Kaggle 会用隐藏测试集重跑本 notebook。
- 若耗时超过 8.5 h，把 `--set features.qwen.batch_size=12` 加上（显存只有 ~3 GB，加到 12 仍安全），或把外部配额降到 2,000/类。
